In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('E-commerce.csv')
df.describe()

,Customer ID,Age,Annual Income,Time on Site
count,50.000000,50.000000,50.000000,50.000000
mean,1004.880000,39.960000,65780.000000,232.597000
std,3.623281,11.067437,17059.667198,109.669736
min,1001.000000,24.000000,40000.000000,32.500000
25%,1002.000000,30.250000,50500.000000,124.100000
50%,1004.000000,37.000000,65000.000000,243.450000
75%,1007.750000,48.000000,80000.000000,300.000000
max,1013.000000,65.000000,100000.000000,486.300000


In [2]:
df.head()

,Customer ID,Age,Gender,Location,Annual Income,Purchase History,Browsing History,Product Reviews,Time on Site
0,1001,25,Female,City D,45000,"[{""Date"": ""2022-03-05"", ""Category"": ""Clothing""...","[{""Timestamp"": ""2022-03-10T14:30:00Z""}, {""Time...","Great pair of jeans, very comfortable. Rating:...",32.50
1,1001,28,Female,City D,52000,"[{""Product Category"": ""Clothing"", ""Purchase Da...","[{""Product Category"": ""Home & Garden"", ""Timest...",Great customer service!,123.45
2,1001,28,Female,City D,65000,"[{""Product Category"": ""Electronics"", ""Purchase...","[{""Product Category"": ""Clothing"", ""Timestamp"":...",Great electronics. The sound quality is excell...,125.60
3,1001,45,Female,City D,70000,"{'Purchase Date': '2022-08-15', 'Product Categ...",{'Timestamp': '2022-09-03 14:30:00'},"{""Product 1"": {""Rating"": 4, ""Review"": ""Great e...",327.60
4,1002,34,Male,City E,45000,"{'Purchase Date': '2022-07-25', 'Product Categ...",{'Timestamp': '2022-08-10 17:15:00'},"{""Product 1"": {""Rating"": 3, ""Review"": ""Good pr...",214.90


In [3]:
df = df.set_index('Customer ID')
df.head()

,Age,Gender,Location,Annual Income,Purchase History,Browsing History,Product Reviews,Time on Site
Customer ID,,,,,,,,
1001,25,Female,City D,45000,"[{""Date"": ""2022-03-05"", ""Category"": ""Clothing""...","[{""Timestamp"": ""2022-03-10T14:30:00Z""}, {""Time...","Great pair of jeans, very comfortable. Rating:...",32.50
1001,28,Female,City D,52000,"[{""Product Category"": ""Clothing"", ""Purchase Da...","[{""Product Category"": ""Home & Garden"", ""Timest...",Great customer service!,123.45
1001,28,Female,City D,65000,"[{""Product Category"": ""Electronics"", ""Purchase...","[{""Product Category"": ""Clothing"", ""Timestamp"":...",Great electronics. The sound quality is excell...,125.60
1001,45,Female,City D,70000,"{'Purchase Date': '2022-08-15', 'Product Categ...",{'Timestamp': '2022-09-03 14:30:00'},"{""Product 1"": {""Rating"": 4, ""Review"": ""Great e...",327.60
1002,34,Male,City E,45000,"{'Purchase Date': '2022-07-25', 'Product Categ...",{'Timestamp': '2022-08-10 17:15:00'},"{""Product 1"": {""Rating"": 3, ""Review"": ""Good pr...",214.90


In [4]:
df["Purchase History"].loc[1004]

Customer ID
1004    {'Purchase Date': '2022-07-20', 'Product Categ...
1004    {'Purchase Date': '2022-04-25', 'Product Categ...
1004    {'Purchase Date': '2022-05-08', 'Product Categ...
1004    [{"Product Category": "Clothing", "Purchase Da...
Name: Purchase History, dtype: object

In [5]:
df.shape

(50, 8)

In [6]:
# data = [df["Purchase History"]]
# data

In [8]:
import ast

keys = set()

for _, item in df.iterrows():
    purchases = item["Purchase History"]
    
    if isinstance(purchases, str):
        purchases = ast.literal_eval(purchases)
    
    if isinstance(purchases, list):
        for p in purchases:
            if isinstance(p, dict):
                keys.update(p.keys())

print(keys)


{'Purchase Date', 'Date', 'Category', 'Price', 'Product Category'}


In [9]:
import ast

keys_2 = set()

for _, item in df.iterrows():
    Browsing = item["Browsing History"]
    
    if isinstance(Browsing, str):
        Browsing = ast.literal_eval(Browsing)
    
    if isinstance(Browsing, list):
        for b in Browsing:
            if isinstance(b, dict):
                keys_2.update(b.keys())

print(keys_2)


{'Timestamp', 'Product Category'}


In [10]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [11]:
for _, item in df.iterrows():
    purchases = item["Purchase History"]
    print(type(purchases))
    print(purchases)
    break


<class 'str'>
[{"Date": "2022-03-05", "Category": "Clothing", "Price": 34.99}, {"Date": "2022-02-12", "Category": "Electronics", "Price": 129.99}, {"Date": "2022-01-20", "Category": "Home & Garden", "Price": 29.99}]


In [12]:
def parse_purchases(purchases):
    """
    Преобразует поле Purchase History / Browsing History в список словарей
    """
    if isinstance(purchases, str):
        try:
            # ast.literal_eval безопасно превращает строку в Python-объект
            return ast.literal_eval(purchases)
        except:
            return []  # если не получилось, возвращаем пустой список
    elif isinstance(purchases, list):
        return purchases
    else:
        return []  # если тип неожиданный


In [13]:
events = []
for _, item in df.iterrows():
    user_id = item.name
    purchases = parse_purchases(item["Purchase History"])
    Browsing = parse_purchases(item["Browsing History"])
    if isinstance(purchases, list):
        for p in purchases:
            if isinstance(p, dict):
                category = None
                for k in p:
                    if "Category" in k:
                        category = p[k]
                        break
                Date = None
                for k in p:
                    print(p.keys())
                    if "Date" in k:
                        Date = p[k]
                        break
                events.append({
                    "user_id": user_id,
                    "category": category,
                    "timestamp": Date,
                    "event_type": "purchase",
                    "Price": p.get("Price")
                })
    if isinstance(Browsing, list):
        for b in Browsing:
            if isinstance(b, dict):
                events.append({
                    "user_id": user_id,
                    "category": b.get('Product Category'),
                    "timestamp": b.get('Timestamp'),
                    "event_type": "view",
                    "Price": None
                })
                
            
    
    

df_events = pd.DataFrame(events)

dict_keys(['Date', 'Category', 'Price'])
dict_keys(['Date', 'Category', 'Price'])
dict_keys(['Date', 'Category', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Purchase Date', 'Product Category', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_keys(['Product Category', 'Purchase Date', 'Price'])
dict_ke

In [14]:
df_events

,user_id,category,timestamp,event_type,Price
0,1001,Clothing,2022-03-05,purchase,34.99
1,1001,Electronics,2022-02-12,purchase,129.99
2,1001,Home & Garden,2022-01-20,purchase,29.99
3,1001,None,2022-03-10T14:30:00Z,view,NaN
4,1001,None,2022-03-11T09:45:00Z,view,NaN
...,...,...,...,...,...
65,1003,Clothing,2022-06-18,purchase,50.00
66,1003,Home & Garden,2022-05-09,purchase,30.00
67,1003,None,2022-08-13T13:45:00Z,view,NaN
68,1003,None,2022-07-26T07:00:00Z,view,NaN


In [15]:
df_events.describe()

,user_id,Price
count,70.000000,34.000000
mean,1001.885714,152.325588
std,1.198343,121.870153
min,1001.000000,25.600000
25%,1001.000000,45.420000
50%,1001.000000,129.990000
75%,1002.750000,237.492500
max,1005.000000,450.000000


In [16]:
df_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   user_id     70 non-null     int64  
 1   category    55 non-null     object 
 2   timestamp   70 non-null     object 
 3   event_type  70 non-null     object 
 4   Price       34 non-null     float64
dtypes: float64(1), int64(1), object(3)
memory usage: 2.9+ KB


In [17]:
df_events["event_type"].value_counts()

event_type
view        36
purchase    34
Name: count, dtype: int64

In [18]:
df_events["category"].value_counts()

category
Electronics      22
Clothing         18
Home & Garden    15
Name: count, dtype: int64

In [19]:
views = df_events[df_events["event_type"] == "view"]
purchases = df_events[df_events["event_type"] == "purchase"]
views_count = views["category"].value_counts()
purchases_count = purchases["category"].value_counts()
conversion = (purchases_count / views_count)
conversion

category
Clothing         2.000000
Electronics      1.444444
Home & Garden    1.500000
Name: count, dtype: float64

**Построение ML**

Поскольку у нас нет product_id поставим цель предсказатьь купит ли пользователь после просмотра по категории

In [20]:
agg = df_events.groupby(["user_id","category"]).agg({
    "event_type": [
        lambda x: (x == "view").sum(),
        lambda x: (x == "purchase").sum()
    ],
    "Price":"mean"
})

agg.columns = ["views","purchases","avg_price"]
agg = agg.reset_index()
agg

,user_id,category,views,purchases,avg_price
0,1001,Clothing,2,7,49.524286
1,1001,Electronics,4,8,212.162500
2,1001,Home & Garden,5,3,114.993333
3,1002,Clothing,2,2,112.600000
4,1002,Electronics,2,2,334.995000
5,1002,Home & Garden,0,3,259.996667
6,1003,Clothing,0,2,42.495000
7,1003,Electronics,1,2,164.995000
8,1003,Home & Garden,1,1,30.000000
9,1004,Clothing,1,1,79.990000


Пишем таргет показывающий "Купил ли пользователь товар после просмотра товара из той же категории(не обязательно что он купил тот же товар что и смотрел)"

In [21]:
agg["target"] = (agg["purchases"] > 0).astype(int)
agg

,user_id,category,views,purchases,avg_price,target
0,1001,Clothing,2,7,49.524286,1
1,1001,Electronics,4,8,212.162500,1
2,1001,Home & Garden,5,3,114.993333,1
3,1002,Clothing,2,2,112.600000,1
4,1002,Electronics,2,2,334.995000,1
5,1002,Home & Garden,0,3,259.996667,1
6,1003,Clothing,0,2,42.495000,1
7,1003,Electronics,1,2,164.995000,1
8,1003,Home & Garden,1,1,30.000000,1
9,1004,Clothing,1,1,79.990000,1


In [22]:
features = ["views", "avg_price"]
x = agg[features]
y = agg["target"]
x["avg_price"] = x["avg_price"].fillna(0)
x

C:\Users\snezk\AppData\Local\Temp\ipykernel_5372\2226908288.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x["avg_price"] = x["avg_price"].fillna(0)


,views,avg_price
0,2,49.524286
1,4,212.162500
2,5,114.993333
3,2,112.600000
4,2,334.995000
5,0,259.996667
6,0,42.495000
7,1,164.995000
8,1,30.000000
9,1,79.990000


In [23]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [24]:
from  sklearn.linear_model import LogisticRegression
from  sklearn.metrics import classification_report
model = LogisticRegression()
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
report = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report

,precision,recall,f1-score,support
1,1.0,1.0,1.0,3.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,3.0
weighted avg,1.0,1.0,1.0,3.0


In [25]:
model.coef_

array([[-1.48376342e-04,  3.25279507e-01]])

In [26]:
agg["conversion_rate"] = agg["purchases"] / (agg["views"]+1)
agg

,user_id,category,views,purchases,avg_price,target,conversion_rate
0,1001,Clothing,2,7,49.524286,1,2.333333
1,1001,Electronics,4,8,212.162500,1,1.600000
2,1001,Home & Garden,5,3,114.993333,1,0.500000
3,1002,Clothing,2,2,112.600000,1,0.666667
4,1002,Electronics,2,2,334.995000,1,0.666667
5,1002,Home & Garden,0,3,259.996667,1,3.000000
6,1003,Clothing,0,2,42.495000,1,2.000000
7,1003,Electronics,1,2,164.995000,1,1.000000
8,1003,Home & Garden,1,1,30.000000,1,0.500000
9,1004,Clothing,1,1,79.990000,1,0.500000


In [27]:
cat_pop = df_events["category"].value_counts()
agg["category_popularity"] = agg["category"].map(cat_pop)
agg

,user_id,category,views,purchases,avg_price,target,conversion_rate,category_popularity
0,1001,Clothing,2,7,49.524286,1,2.333333,18
1,1001,Electronics,4,8,212.162500,1,1.600000,22
2,1001,Home & Garden,5,3,114.993333,1,0.500000,15
3,1002,Clothing,2,2,112.600000,1,0.666667,18
4,1002,Electronics,2,2,334.995000,1,0.666667,22
5,1002,Home & Garden,0,3,259.996667,1,3.000000,15
6,1003,Clothing,0,2,42.495000,1,2.000000,18
7,1003,Electronics,1,2,164.995000,1,1.000000,22
8,1003,Home & Garden,1,1,30.000000,1,0.500000,15
9,1004,Clothing,1,1,79.990000,1,0.500000,18


In [28]:
from sklearn.ensemble import RandomForestClassifier
model_1 = RandomForestClassifier()
model_1.fit(x_train, y_train)
y_pred = model_1.predict(x_test)
report_1 = classification_report(y_test, y_pred, output_dict=True)
df_report_1 = pd.DataFrame(report_1).transpose()
df_report_1

,precision,recall,f1-score,support
1,1.0,1.0,1.0,3.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,3.0
weighted avg,1.0,1.0,1.0,3.0


In [29]:
print(x.columns)
print(y.value_counts())
print(len(x_train), len(x_test))

Index(['views', 'avg_price'], dtype='object')
target
1    13
0     2
Name: count, dtype: int64
12 3


**Вывод**
Слишком мало данных для обучения модели она просто запаминает

**Новая задача**

предсказать приведёт ли просмотр к покупке

In [30]:
views = df_events[df_events["event_type"] == "view"].copy()
purchases = df_events[df_events["event_type"] == "purchase"].copy()
purchases_pairs = set(zip(purchases["user_id"], purchases["category"]))
views["target"] = views.apply(lambda x: (x["user_id"], x["category"]) in purchases_pairs, axis = 1).astype(int)
views

,user_id,category,timestamp,event_type,Price,target
3,1001,None,2022-03-10T14:30:00Z,view,NaN,0
4,1001,None,2022-03-11T09:45:00Z,view,NaN,0
5,1001,None,2022-03-12T16:20:00Z,view,NaN,0
8,1001,Home & Garden,2022-05-12T13:30:00Z,view,NaN,1
9,1001,Electronics,2022-06-05T08:45:00Z,view,NaN,1
12,1001,Clothing,2022-09-03 15:30:00,view,NaN,1
13,1001,Electronics,2022-09-01 10:15:00,view,NaN,1
15,1001,None,2022-08-20T14:30:00Z,view,NaN,0
16,1001,None,2022-08-22T16:45:00Z,view,NaN,0
17,1001,None,2022-09-03T10:20:00Z,view,NaN,0


In [31]:
purchases

,user_id,category,timestamp,event_type,Price
0,1001,Clothing,2022-03-05,purchase,34.99
1,1001,Electronics,2022-02-12,purchase,129.99
2,1001,Home & Garden,2022-01-20,purchase,29.99
6,1001,Clothing,2022-05-15,purchase,34.56
7,1001,Electronics,2022-06-02,purchase,150.99
10,1001,Electronics,2022-08-15,purchase,345.50
11,1001,Home & Garden,2022-07-02,purchase,279.99
14,1001,Clothing,2022-09-15,purchase,34.50
18,1001,Clothing,2022-08-15,purchase,45.23
19,1001,Electronics,2022-07-20,purchase,250.34


In [32]:
views["timestamp"] =pd.to_datetime(views["timestamp"], errors="coerce")
views["hour"] = pd.to_datetime(views["timestamp"]).dt.hour
views["day"] = pd.to_datetime(views["timestamp"]).dt.dayofweek

In [33]:
views["category_freq"] = views["category"].map(df_events["category"].value_counts())

In [34]:
features = ["hour", "day", "category_freq"]
x = views[features]
y = views["target"]

In [35]:
x=pd.get_dummies(x.join(views["category"]), columns = ["category"])
x

,hour,day,category_freq,category_Clothing,category_Electronics,category_Home & Garden
3,14.0,3.0,NaN,False,False,False
4,9.0,4.0,NaN,False,False,False
5,16.0,5.0,NaN,False,False,False
8,13.0,3.0,15.0,False,False,True
9,8.0,6.0,22.0,False,True,False
12,NaN,NaN,18.0,True,False,False
13,NaN,NaN,22.0,False,True,False
15,14.0,5.0,NaN,False,False,False
16,16.0,0.0,NaN,False,False,False
17,10.0,5.0,NaN,False,False,False


In [38]:
x["category_Clothing"] = x["category_Clothing"].astype(int)
x["category_Electronics"] = x["category_Electronics"].astype(int)
x["category_Home & Garden"] = x["category_Home & Garden"].astype(int)

In [44]:
x = x.fillna(0)
x

,hour,day,category_freq,category_Clothing,category_Electronics,category_Home & Garden
3,14.0,3.0,0.0,0,0,0
4,9.0,4.0,0.0,0,0,0
5,16.0,5.0,0.0,0,0,0
8,13.0,3.0,15.0,0,0,1
9,8.0,6.0,22.0,0,1,0
12,0.0,0.0,18.0,1,0,0
13,0.0,0.0,22.0,0,1,0
15,14.0,5.0,0.0,0,0,0
16,16.0,0.0,0.0,0,0,0
17,10.0,5.0,0.0,0,0,0


In [47]:
x_train,x_test,y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:,1]
report = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report)
df_report

,0,1,accuracy,macro avg,weighted avg
precision,1.0,1.0,1.0,1.0,1.0
recall,1.0,1.0,1.0,1.0,1.0
f1-score,1.0,1.0,1.0,1.0,1.0
support,3.0,5.0,1.0,8.0,8.0


from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc

Данных слишком мало для точной модели